### Importando Bibliotecas

In [4]:
import sqlite3
import pandas as pd

### Conexão ao banco de dados

In [5]:
conexao = sqlite3.connect("Dados/ecommerce.db")

print("Conexão realizada!")

cursor = conexao.cursor()

cursor.execute("""
    SELECT name
    FROM sqlite_master
    WHERE type = 'table';
""")

tabelas = cursor.fetchall()

print(tabelas)

Conexão realizada!
[('categorias',), ('produtos',), ('clientes',), ('pedidos',), ('itens_pedido',)]


### Células de Exploração e Limpeza de Dados

In [6]:
pedidos = pd.read_sql_query(
    "SELECT * FROM pedidos",
    conexao
)



In [7]:
pedidos.head()

,id_pedido,id_cliente,data_pedido,status,forma_pagamento,frete
0,1,1,2026-05-26,Entregue,Pix,19.9
1,2,2,2026-01-12,Entregue,Pix,14.9
2,3,3,2026-02-23,Entregue,Cartão de Crédito,24.9
3,4,4,2025-12-27,Entregue,Pix,19.9
4,5,5,2025-02-07,Entregue,Pix,19.9


In [8]:
pedidos.duplicated().sum()

np.int64(0)

In [9]:
pedidos['data_pedido'] = pd.to_datetime(pedidos['data_pedido'])

In [10]:
pedidos.info()

<class 'pandas.DataFrame'>
RangeIndex: 1312 entries, 0 to 1311
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   id_pedido        1312 non-null   int64         
 1   id_cliente       1312 non-null   int64         
 2   data_pedido      1312 non-null   datetime64[us]
 3   status           1312 non-null   str           
 4   forma_pagamento  1312 non-null   str           
 5   frete            1312 non-null   float64       
dtypes: datetime64[us](1), float64(1), int64(2), str(2)
memory usage: 88.0 KB


In [11]:
pedidos["status"].value_counts()

status
Entregue     1128
Cancelado      81
Enviado        70
Devolvido      33
Name: count, dtype: int64

In [12]:
pedidos['forma_pagamento'].value_counts()

forma_pagamento
Cartão de Crédito    654
Pix                  397
Boleto               156
Cartão de Débito     105
Name: count, dtype: int64

In [13]:
pedidos["frete"].describe()

count    1312.000000
mean       13.865396
std         8.711644
min         0.000000
25%         9.900000
50%        14.900000
75%        19.900000
max        24.900000
Name: frete, dtype: float64

In [14]:
(pedidos["frete"]==0).sum()

np.int64(276)

In [15]:
pedidos.groupby("status")["frete"].agg(["count", "mean", "min", "max"])

,count,mean,min,max
status,,,,
Cancelado,81,14.729630,0.0,24.9
Devolvido,33,13.406061,0.0,24.9
Entregue,1128,13.755762,0.0,24.9
Enviado,70,14.848571,0.0,24.9


In [16]:
pedidos[pedidos["frete"] == 0].groupby("status").size()

status
Cancelado     12
Devolvido      7
Entregue     243
Enviado       14
dtype: int64

### Células de exploração de clientes


In [17]:
clientes = pd.read_sql_query(
    'SELECT * FROM clientes',
    conexao
)

In [18]:
clientes.head()

,id_cliente,nome,email,cidade,estado,data_cadastro
0,1,Larissa Ribeiro,larissa.ribeiro1@email.com,BELO HORIZONTE,MG,2026-05-23
1,2,Rodrigo Pereira,rodrigo.pereira2@email.com,Salvador,BA,2024-08-21
2,3,Matheus Souza,matheus.souza3@email.com,Rio de Janeiro,RJ,2025-09-08
3,4,Lucas Ferreira,lucas.ferreira4@email.com,Rio de Janeiro,RJ,2025-01-01
4,5,Juliana Oliveira,juliana.oliveira5@email.com,Uberlândia,MG,2024-11-23


In [19]:
clientes.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   id_cliente     500 non-null    int64
 1   nome           500 non-null    str  
 2   email          494 non-null    str  
 3   cidade         496 non-null    str  
 4   estado         500 non-null    str  
 5   data_cadastro  500 non-null    str  
dtypes: int64(1), str(5)
memory usage: 53.9 KB


In [20]:
clientes['data_cadastro'] = pd.to_datetime(clientes['data_cadastro'])

In [21]:
clientes.isna().sum()

id_cliente       0
nome             0
email            6
cidade           4
estado           0
data_cadastro    0
dtype: int64

In [22]:
clientes['cidade'].value_counts()

cidade
São Paulo         99
Rio de Janeiro    81
Belo Horizonte    38
Fortaleza         34
Porto Alegre      31
Salvador          30
Curitiba          30
Campinas          30
Recife            21
Brasília          19
Uberlândia        18
Goiânia           18
Niterói           15
Florianópolis     10
Manaus            10
BELO HORIZONTE     3
RIO DE JANEIRO     3
RECIFE             2
CURITIBA           2
SÃO PAULO          1
FORTALEZA          1
Name: count, dtype: int64

In [23]:
#Replace em cidade

clientes['cidade'] = clientes['cidade'].replace({

    "BELO HORIZONTE": "Belo Horizonte",
    "RIO DE JANEIRO": "Rio de Janeiro",
    "RECIFE": "Recife",
    "CURITIBA": "Curitiba",
    "SÃO PAULO": "São Paulo",
    "FORTALEZA": "Fortaleza"
})

In [24]:
clientes['cidade'].value_counts()

cidade
São Paulo         100
Rio de Janeiro     84
Belo Horizonte     41
Fortaleza          35
Curitiba           32
Porto Alegre       31
Salvador           30
Campinas           30
Recife             23
Brasília           19
Uberlândia         18
Goiânia            18
Niterói            15
Florianópolis      10
Manaus             10
Name: count, dtype: int64

In [25]:
clientes["email"].isna().sum()

np.int64(6)

In [26]:
clientes["nome"].isna().sum()

np.int64(0)

In [27]:
clientes["nome"].duplicated().sum()

np.int64(168)

In [28]:
clientes[clientes["nome"].duplicated(keep=False)][["id_cliente", "nome"]].sort_values("nome")

,id_cliente,nome
18,19,Ana Lima
173,174,Ana Lima
283,284,Ana Lopes
392,393,Ana Lopes
30,31,Ana Oliveira
...,...,...
50,51,Vitor Santos
160,161,Vitor Santos
277,278,Vitor Santos
308,309,Vitor Santos


In [29]:
clientes["estado"].value_counts()

estado
SP    131
RJ     97
MG     59
CE     33
PR     32
RS     31
BA     30
PE     21
DF     19
GO     19
SC     10
AM     10
rj      3
pe      2
ce      2
mg      1
Name: count, dtype: int64

In [30]:
clientes['estado'] = clientes['estado'].str.upper()

In [31]:
clientes["estado"].value_counts()

estado
SP    131
RJ    100
MG     60
CE     35
PR     32
RS     31
BA     30
PE     23
DF     19
GO     19
SC     10
AM     10
Name: count, dtype: int64

In [32]:
#Análise de Cadastro por mês
round(clientes["data_cadastro"].dt.to_period("M").value_counts().mean(), 2)

np.float64(18.52)

### Células de exploração e limpeza de produtos

In [33]:
produtos = pd.read_sql_query("SELECT * FROM produtos", conexao)
produtos.info()

<class 'pandas.DataFrame'>
RangeIndex: 36 entries, 0 to 35
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id_produto    36 non-null     int64  
 1   nome          36 non-null     str    
 2   id_categoria  36 non-null     int64  
 3   preco         36 non-null     float64
 4   custo         36 non-null     float64
dtypes: float64(2), int64(2), str(1)
memory usage: 2.1 KB


In [34]:
produtos.head()

,id_produto,nome,id_categoria,preco,custo
0,1,Fone Bluetooth,1,149.9,91.42
1,2,Carregador Turbo 20W,1,89.9,41.02
2,3,Smartwatch Fit,1,399.0,206.98
3,4,Caixa de Som Portátil,1,219.9,111.23
4,5,Mouse Sem Fio,1,69.9,44.32


In [35]:
produtos['id_produto'].duplicated().sum()

np.int64(0)

In [36]:
(produtos['custo']> produtos['preco']).sum()
(produtos["preco"] <= 0).sum()

np.int64(0)

In [37]:
produtos['lucro_bruto'] = produtos['preco']- produtos['custo']

In [38]:
produtos["margem_bruta"] = (
    produtos["lucro_bruto"] / produtos["preco"]
) * 100

produtos["margem_bruta_formatada"] = (
    produtos["margem_bruta"].map("{:.2f}%".format)
)

In [39]:
produtos.isna().sum()

id_produto                0
nome                      0
id_categoria              0
preco                     0
custo                     0
lucro_bruto               0
margem_bruta              0
margem_bruta_formatada    0
dtype: int64

In [41]:
itens_pedido = pd.read_sql_query("SELECT * FROM itens_pedido", conexao)
itens_pedido.info()

<class 'pandas.DataFrame'>
RangeIndex: 2285 entries, 0 to 2284
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id_item         2285 non-null   int64  
 1   id_pedido       2285 non-null   int64  
 2   id_produto      2285 non-null   int64  
 3   quantidade      2285 non-null   int64  
 4   preco_unitario  2285 non-null   float64
 5   desconto        2285 non-null   float64
dtypes: float64(2), int64(4)
memory usage: 107.2 KB


In [42]:
print("IDs de item duplicados:", itens_pedido["id_item"].duplicated().sum())
print("Preços inválidos:", (itens_pedido["preco_unitario"] <= 0).sum())
print("Descontos negativos:", (itens_pedido["desconto"] < 0).sum())
print("Descontos acima de 100%:", (itens_pedido["desconto"] > 1).sum())

IDs de item duplicados: 0
Preços inválidos: 0
Descontos negativos: 0
Descontos acima de 100%: 0


In [43]:
print("Pedidos inexistentes:", (~itens_pedido["id_pedido"].isin(pedidos["id_pedido"])).sum())
print("Produtos inexistentes:", (~itens_pedido["id_produto"].isin(produtos["id_produto"])).sum())

Pedidos inexistentes: 0
Produtos inexistentes: 0


In [45]:
categorias = pd.read_sql_query('SELECT * FROM categorias', conexao)

In [46]:
print("Pedidos:", pedidos.shape)
print("Clientes:", clientes.shape)
print("Produtos:", produtos.shape)
print("Categorias:", categorias.shape)
print("Itens dos pedidos:", itens_pedido.shape)

Pedidos: (1312, 6)
Clientes: (500, 6)
Produtos: (36, 8)
Categorias: (6, 2)
Itens dos pedidos: (2285, 6)


### Exportação dos CSVs para o Looker Studio

In [47]:
import os

os.makedirs("Dados/csv", exist_ok=True)

pedidos.to_csv("Dados/csv/pedidos.csv", index=False)
clientes.to_csv("Dados/csv/clientes.csv", index=False)
produtos.to_csv("Dados/csv/produtos.csv", index=False)
categorias.to_csv("Dados/csv/categorias.csv", index=False)
itens_pedido.to_csv("Dados/csv/itens_pedido.csv", index=False)

### Consultas SQL utilizadas no projeto

In [48]:
# Faturamento mensal

consulta_faturamento = """
SELECT
    strftime('%Y-%m-01', p.data_pedido) AS mes,
    ROUND(
        SUM(i.quantidade * i.preco_unitario * (1 - i.desconto)),
        2
    ) AS faturamento
FROM itens_pedido AS i
JOIN pedidos AS p
    ON p.id_pedido = i.id_pedido
WHERE p.status = 'Entregue'
GROUP BY strftime('%Y-%m', p.data_pedido)
ORDER BY mes;
"""

faturamento_mensal = pd.read_sql_query(
    consulta_faturamento, conexao
)

faturamento_mensal.to_csv(
    "Dados/csv/faturamento_mensal.csv",
    index=False
)

display(faturamento_mensal.head())

,mes,faturamento
0,2024-07-01,399.00
1,2024-08-01,989.40
2,2024-09-01,1828.48
3,2024-10-01,1469.78
4,2024-11-01,3182.85


In [49]:
#Total de vendas por produto

consulta = """
SELECT
    t2.nome AS produto,
    SUM(t1.quantidade) AS unidades_vendidas
FROM itens_pedido AS t1
INNER JOIN produtos AS t2
    ON t1.id_produto = t2.id_produto
INNER JOIN pedidos AS t3
    ON t1.id_pedido = t3.id_pedido
WHERE t3.status = 'Entregue'
GROUP BY t2.id_produto, t2.nome
ORDER BY SUM(t1.quantidade) DESC;
"""

df_produtos_vendidos = pd.read_sql_query(consulta, conexao)

df_produtos_vendidos.to_csv(
    "Dados/csv/produtos_mais_vendidos.csv",
    index=False,
    encoding="utf-8-sig"
)


display(df_produtos_vendidos.head(10))

,produto,unidades_vendidas
0,Tênis Casual,106
1,Smartwatch Fit,102
2,Mouse Sem Fio,98
3,Boné,97
4,Mochila Urbana,96
5,Calça Jeans,94
6,Camiseta Básica,91
7,Caixa de Som Portátil,89
8,Perfume 100ml,87
9,Air Fryer 4L,85


In [50]:

# Total de vendas por categoria

consulta = """
SELECT 
    t3.nome AS nome_categoria,
    ROUND(
        SUM(t1.quantidade * t1.preco_unitario * (1 - t1.desconto)),
        2
    ) AS Faturamento
FROM itens_pedido AS t1
INNER JOIN produtos AS t2
    ON t1.id_produto = t2.id_produto
INNER JOIN categorias AS t3
    ON t2.id_categoria = t3.id_categoria
INNER JOIN pedidos AS t4
    ON t1.id_pedido = t4.id_pedido
WHERE t4.status = 'Entregue'
GROUP BY t3.id_categoria, t3.nome
ORDER BY Faturamento DESC;
"""

df_categoria = pd.read_sql_query(consulta, conexao)

df_categoria.to_csv(
    "Dados/csv/faturamento_categoria.csv",
    index=False,
    encoding="utf-8-sig"
)

display(df_categoria)


,nome_categoria,Faturamento
0,Eletrônicos,104834.38
1,Casa e Cozinha,77312.20
2,Moda,70584.22
3,Beleza,55353.39
4,Esporte,16876.40
5,Livros,16670.60


In [51]:
# Diferença entre preço e custo

# Conectar ao banco correto
conexao = sqlite3.connect("Dados/ecommerce.db")

consulta = """
SELECT
    nome AS produto,
    preco,
    custo,
    preco - custo AS diferenca
FROM produtos
ORDER BY diferenca DESC;
"""

# Executar a consulta e carregar os dados
df_diferenca = pd.read_sql_query(consulta, conexao)

# Exportar o resultado para CSV
df_diferenca.to_csv(
    "Dados/csv/diferenca_preco_custo.csv",
    index=False,
    encoding="utf-8-sig"
)

display(df_diferenca.head(10))

conexao.close()

,produto,preco,custo,diferenca
0,Smartwatch Fit,399.0,206.98,192.02
1,Jogo de Panelas,259.9,122.60,137.30
2,Air Fryer 4L,349.9,235.50,114.40
3,Teclado Mecânico,289.9,179.50,110.40
4,Caixa de Som Portátil,219.9,111.23,108.67
5,Cafeteira Elétrica,199.9,91.44,108.46
6,Tênis Casual,229.9,140.81,89.09
7,Jaqueta Corta-Vento,189.9,111.33,78.57
8,Liquidificador,159.9,88.82,71.08
9,Calça Jeans,139.9,69.91,69.99


In [52]:
# Pedidos por estado

conexao = sqlite3.connect("Dados/ecommerce.db")

# Consulta SQL que já fizemos
consulta = """
SELECT
    UPPER(t2.estado) AS estado,
    COUNT(t1.id_pedido) AS total_pedidos
FROM pedidos AS t1
LEFT JOIN clientes AS t2
    ON t1.id_cliente = t2.id_cliente
GROUP BY UPPER(t2.estado)
ORDER BY COUNT(t1.id_pedido) DESC;
"""

# Carregar os resultados no Pandas
df_pedidos_estado = pd.read_sql_query(consulta, conexao)

# Exportar para CSV
df_pedidos_estado.to_csv(
    "Dados/csv/pedidos_por_estado.csv",
    index=False,
    encoding="utf-8-sig"
)

# Visualizar os resultados
display(df_pedidos_estado)

,estado,total_pedidos
0,SP,342
1,RJ,259
2,MG,149
3,CE,99
4,BA,97
5,PR,86
6,RS,81
7,DF,51
8,PE,50
9,GO,46


In [53]:
#Clientes por estado

consulta = """
SELECT
    UPPER(estado) AS estado,
    COUNT(id_cliente) AS total_clientes
FROM clientes
GROUP BY UPPER(estado)
ORDER BY COUNT(id_cliente) DESC;
"""

# Carregar os dados no Pandas
df_clientes_estado = pd.read_sql_query(consulta, conexao)

# Exportar para CSV
df_clientes_estado.to_csv(
    "Dados/csv/clientes_por_estado.csv",
    index=False,
    encoding="utf-8-sig"
)

# Visualizar o resultado
display(df_clientes_estado)

,estado,total_clientes
0,SP,131
1,RJ,100
2,MG,60
3,CE,35
4,PR,32
5,RS,31
6,BA,30
7,PE,23
8,GO,19
9,DF,19
